In [ ]:

# Enrichment Score calculation based on TIC values from pellet and supernatant (triplicates). Input data is de novo sequencing results output from PEAKS 13.
# E = (pellet/((supernatant + pellet))*100

# =========================
# Imports
# =========================

import pandas as pd
import numpy as np
import re
from pathlib import Path
import matplotlib.pyplot as plt
import logomaker
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import FixedLocator

# =========================
# Configuration
# =========================

BASE_DIR = Path(r".\data")
PELLET_DIR = BASE_DIR / "pellet"
SUPERNATANT_DIR = BASE_DIR / "supernatant"

OUTPUT_DIR = BASE_DIR / "enrichment_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / "Enrichment_Score.csv"

SAMPLE_PREFIX = "example_library"
REPLICATES = [1, 2, 3]

INPUT_FILENAME = "filtered-peptides-allowedletters.csv"

# =========================
# Helper functions
# =========================

def is_valid_peptide(peptide):
    cleaned = re.sub(r"\(.*?\)", "", peptide)
    return all(aa in {"W", "R", "N", "D", "K"} for aa in cleaned) # replace with amino acids that were used in variable positions

def load_filter_sum(filepath, new_column_name):
    df = pd.read_csv(filepath)

    df = (
        df.groupby("Peptide", as_index=False)
          .agg({"Area": "sum"})
          .rename(columns={"Area": new_column_name})
    )
    return df

# =========================
# Load pellet files
# =========================

pellet_dfs = []
for rep in REPLICATES:
    folder = f"{SAMPLE_PREFIX}-{rep}"
    filepath = PELLET_DIR / folder / INPUT_FILENAME
    col_name = f"Pellet_{rep}"
    pellet_dfs.append(load_filter_sum(filepath, col_name))

# =========================
# Load supernatant files
# =========================

supernatant_dfs = []
for rep in REPLICATES:
    folder = f"{SAMPLE_PREFIX}-{rep}"
    filepath = SUPERNATANT_DIR / folder / INPUT_FILENAME
    col_name = f"Supernatant_{rep}"
    supernatant_dfs.append(load_filter_sum(filepath, col_name))

# =========================
# Merge all on Peptide
# =========================

merged_df = pellet_dfs[0]
for df in pellet_dfs[1:] + supernatant_dfs:
    merged_df = merged_df.merge(df, on="Peptide", how="outer")

merged_df.fillna(0, inplace=True)

# =========================
# Calculations
# =========================

super_cols = [f"Supernatant_{rep}" for rep in REPLICATES]
merged_df["Avg_Area_Supernatant"] = merged_df[super_cols].mean(axis=1)

for rep in REPLICATES:
    merged_df[f"Ratio_{rep}"] = (merged_df[f"Pellet_{rep}"] / (merged_df["Avg_Area_Supernatant"]+ merged_df[f"Pellet_{rep}"]))*100

diff_cols = [f"Ratio_{rep}" for rep in REPLICATES]
merged_df["Avg_Ratio_Area"] = merged_df[diff_cols].mean(axis=1)
merged_df["Std_Ratio_Area"] = merged_df[diff_cols].std(axis=1)

for rep in REPLICATES:
    merged_df[f"Sub_{rep}"] = merged_df[f"Pellet_{rep}"] - merged_df["Avg_Area_Supernatant"]

sub_cols = [f"Sub_{rep}" for rep in REPLICATES]
merged_df["Avg_Area_Sub"] = merged_df[sub_cols].mean(axis=1)
merged_df["Std_Area_Sub"] = merged_df[sub_cols].std(axis=1)

# =========================
# Clean sequence column
# =========================

merged_df["Clean_Sequence"] = merged_df["Peptide"].str.replace(
    r"\(.*?\)", "", regex=True
)

# =========================
# Group assignment
# =========================

def assign_group(x):
    if x == 100:
        return "100"
    elif 10 <= x <= 99.99:
        return "10-99"
    elif x == 0:
        return "0"
    else:
        return "other"

merged_df["Group"] = merged_df["Avg_Ratio_Area"].apply(assign_group)

group1 = merged_df[merged_df["Group"] == "100"]
group2 = merged_df[merged_df["Group"] == "10-99"]
group3 = merged_df[merged_df["Group"] == "0"]

# =========================
# Save output CSV
# =========================

merged_df_sorted = merged_df.sort_values(
    by="Avg_Ratio_Area", ascending=False
)

output_cols = [
    "Peptide",
    "Clean_Sequence",
    "Group",
    "Avg_Ratio_Area", "Std_Ratio_Area",
    "Ratio_1", "Ratio_2", "Ratio_3",
    "Avg_Area_Sub", "Std_Area_Sub",
    "Sub_1", "Sub_2", "Sub_3",
]

merged_df_sorted.to_csv(
    OUTPUT_FILE, columns=output_cols, index=False
)

print(f"Results saved to {OUTPUT_FILE}")

# =========================
# WebLogo plotting
# =========================

AA_COLORS = {
    # Hydrophobic
    'W': "#560030", 'F': "#B50D6C", 'Y': "#CE4693",
    'A': '#E66101', 'V': "#A34A0A", 'L': "#7D3908", 'I': "#5B2804",
    'K': "#1978AF", 'R': "#032740", 'H': "#76BDE7",
    'D': '#7C131C', 'E': "#DA4760",
    'S': '#1A9850', 'T': "#26C46B", 'N': "#085719", 'Q': "#A7F0C7",
    'G': '#CCCCCC', 'P': '#CCCCCC', 'C': '#CCCCCC', 'M': '#CCCCCC'
}

def plot_weblogo(df, group_name, output_dir):
    sequences = df["Clean_Sequence"].dropna().tolist()

    if len(sequences) == 0:
        print(f"No peptides in group {group_name}")
        return

    counts_df = logomaker.alignment_to_matrix(
        sequences=sequences,
        to_type="counts"
    )

    freq_df = counts_df.div(counts_df.sum(axis=1), axis=0)

    fig, ax = plt.subplots(figsize=(6, 6))

    logomaker.Logo(
        freq_df,
        ax=ax,
        color_scheme=AA_COLORS
    )

    '''
    ax.set_xlabel("Position")
    ax.set_ylabel("Frequency")
    ax.set_title(f"WebLogo – Avg_Ratio_Area {group_name}")
    '''
    ax.set_ylabel("Probability", fontsize=44) #fontweight='bold'
    ax.set_xlabel("Position", fontsize=44) #, fontweight='bold'
    ax.set_title(f"WebLogo – Avg_Ratio_Area {group_name}", fontsize=13, fontweight='bold', y=1.1)
    ax.set_ylim(0, 1.0)
        # Y-axis: max 3 ticks (will show 0, 0.5, 1)
    ax.yaxis.set_major_locator(FixedLocator([0, 0.5, 1.0]))
    # X-axis: only integer ticks
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.tick_params(axis='both', which='major', labelsize=36)

    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

    # Peptide count OUTSIDE logo
    fig.text(
        0.99, 0.99,
        f"n = {len(sequences)}",
        ha="right",
        va="top",
        fontsize=8
    )

    plt.tight_layout()
    plt.savefig(
        output_dir / f"{SAMPLE_PREFIX}_ratio_weblogo_group_{group_name}.png",
        dpi=300
    )
    plt.close()

# =========================
# Generate WebLogos
# =========================

plot_weblogo(group1, "100", OUTPUT_DIR)
plot_weblogo(group2, "10-99", OUTPUT_DIR)
plot_weblogo(group3, "0", OUTPUT_DIR)

print("WebLogo plots generated.")


# =========================
# Excel summary by group
# =========================

EXCEL_SUMMARY_FILE = OUTPUT_DIR / f"{SAMPLE_PREFIX}_Enrichment_Score-summary.xlsx"

# Columns to include in ranking sheets
summary_cols = [
    "Peptide",
    "Clean_Sequence",
    "Avg_Ratio_Area",
    "Std_Ratio_Area",
    "Ratio_1", "Ratio_2", "Ratio_3",
    "Avg_Area_Sub",
    "Std_Area_Sub",
]

with pd.ExcelWriter(EXCEL_SUMMARY_FILE, engine="xlsxwriter") as writer:

    for group_name, group_df in {
        "100": group1,
        "10-99": group2,
        "0": group3,
        "other": merged_df[merged_df["Group"] == "other"],
    }.items():

        if group_df.empty:
            continue

        ranked_df = (
            group_df
            .sort_values("Avg_Ratio_Area", ascending=False)
            .loc[:, summary_cols]
            .reset_index(drop=True)
        )

        # Add explicit ranking column
        ranked_df.insert(0, "Rank", ranked_df.index + 1)

        ranked_df.to_excel(
            writer,
            sheet_name=group_name,
            index=False
        )

print(f"Excel summary saved to {EXCEL_SUMMARY_FILE}")



